# Data Sharding & Efficient Pipeline Demonstration with MNIST

This notebook demonstrates the end-to-end workflow of:
1. Understanding the non-sharded file I/O bottleneck by loading individual image files.
2. Creating discrete **data shards** aligned with batch boundaries.
3. Profiling I/O speed improvements from data sharded formats.
4. Implementing a custom PyTorch `IterableDataset` with **multi-worker safety** and **in-memory buffer shuffling**.
5. Extracting streaming data from shards to train and evaluate a **Support Vector Machine (SVM)** model.

To download this notebook click on this [link](https://github.com/SETU-AIML-2026/DH-Infra-AI/blob/main/topic-02-data-storage/unit-2-data-storage/notebook-2-data-shards/notebook-shards.ipynb)

In [ ]:
import os
import time
import glob
import random
import numpy as np
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import IterableDataset, DataLoader
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

# Set up local directories
BASE_DIR = "./mnist_sharding_demo"
INDIVIDUAL_FILES_DIR = os.path.join(BASE_DIR, "individual_files")
SHARDS_DIR = os.path.join(BASE_DIR, "shards")

os.makedirs(INDIVIDUAL_FILES_DIR, exist_ok=True)
os.makedirs(SHARDS_DIR, exist_ok=True)

print("Environment setup complete.")

## Step 1: Download MNIST & Save as Individual Files
Simulating standard un-sharded dataset structures where thousands of small files are stored individually on disk.

In [ ]:
# Load standard MNIST training dataset
mnist_raw = datasets.MNIST(root=BASE_DIR, train=True, download=True, transform=transforms.ToTensor())

# Save 10,000 samples as individual .pt files
NUM_SAMPLES = 10000
print(f"Saving {NUM_SAMPLES} samples as individual files to disk...")

for idx in range(NUM_SAMPLES):
    image, label = mnist_raw[idx]
    file_path = os.path.join(INDIVIDUAL_FILES_DIR, f"sample_{idx:05d}.pt")
    torch.save({"image": image, "label": label}, file_path)

print("Finished writing individual files.")

Now let's take a look at the data!

In [ ]:
import matplotlib.pyplot as plt

# Create a figure grid for 10 images (2 rows, 5 columns)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.flat):
    image, label = mnist_raw[i]
    
    # Remove single-channel dimension: (1, 28, 28) -> (28, 28)
    image_sq = image.squeeze()
    
    ax.imshow(image_sq, cmap='gray')
    ax.set_title(f"Label: {label}")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 2: Profiling Individual Files Loading Time
Timing sequential disk reads across individual files.

In [ ]:
def load_individual_files(file_dir, count):
    loaded_data = []
    for idx in range(count):
        file_path = os.path.join(file_dir, f"sample_{idx:05d}.pt")
        data = torch.load(file_path)
        loaded_data.append(data)
    return loaded_data

start_time = time.perf_counter()
individual_data = load_individual_files(INDIVIDUAL_FILES_DIR, NUM_SAMPLES)
end_time = time.perf_counter()

individual_loading_time = end_time - start_time
print(f"Time to load {NUM_SAMPLES} individual files: {individual_loading_time:.4f} seconds")

## Step 3: Create 3 Shards Aligned with Batch Boundaries
Packing individual samples into 3 shard files such that full batches are kept intact without splitting across boundaries.

In [ ]:
NUM_SHARDS = 3
BATCH_SIZE = 64

# Calculate batch alignments
total_batches = NUM_SAMPLES // BATCH_SIZE
batches_per_shard = total_batches // NUM_SHARDS
aligned_total_samples = batches_per_shard * NUM_SHARDS * BATCH_SIZE

print(f"Total Aligned Samples: {aligned_total_samples}")
print(f"Batches per Shard: {batches_per_shard} (Batch size: {BATCH_SIZE})")

# Write to shards
for shard_idx in range(NUM_SHARDS):
    shard_batches = []
    
    start_batch_idx = shard_idx * batches_per_shard
    end_batch_idx = start_batch_idx + batches_per_shard
    
    for batch_i in range(start_batch_idx, end_batch_idx):
        batch_images = []
        batch_labels = []
        for item_i in range(BATCH_SIZE):
            sample_idx = (batch_i * BATCH_SIZE) + item_i
            file_path = os.path.join(INDIVIDUAL_FILES_DIR, f"sample_{sample_idx:05d}.pt")
            sample = torch.load(file_path)
            batch_images.append(sample["image"])
            batch_labels.append(sample["label"])
            
        shard_batches.append({
            "images": torch.stack(batch_images),
            "labels": torch.tensor(batch_labels)
        })
    
    shard_path = os.path.join(SHARDS_DIR, f"mnist_shard_{shard_idx}.pt")
    torch.save(shard_batches, shard_path)
    print(f"Saved Shard {shard_idx} -> {shard_path}")

print("Sharding complete.")

## Step 4: Profiling Sharded Data Loading Time & Comparison
Loading data sequentially from the 3 shard files and measuring overall performance gain.

In [ ]:
start_time = time.perf_counter()

sharded_data = []
shard_files = sorted(glob.glob(os.path.join(SHARDS_DIR, "mnist_shard_*.pt")))

for shard_file in shard_files:
    shard_content = torch.load(shard_file)
    sharded_data.extend(shard_content)

end_time = time.perf_counter()

sharded_loading_time = end_time - start_time
speedup = individual_loading_time / sharded_loading_time

print("================ Profiling Summary ================")
print(f"Individual Files Load Time : {individual_loading_time:.4f} seconds")
print(f"Sharded Files Load Time    : {sharded_loading_time:.4f} seconds")
print(f"Speedup Factor             : {speedup:.2f}x faster")
print("===================================================")

## Step 5: Custom PyTorch IterableDataset with Buffer Shuffling
Designing a custom dataset that stream-loads shards safely across multiple PyTorch loader workers and shuffles samples using an in-memory buffer.

In [ ]:
class BufferedShardedDataset(IterableDataset):
    def __init__(self, shard_paths, shuffle_shards=True, buffer_size=1000):
        super().__init__()
        self.shard_paths = shard_paths
        self.shuffle_shards = shuffle_shards
        self.buffer_size = buffer_size

    def _sample_generator(self, assigned_shards):
        if self.shuffle_shards:
            assigned_shards = list(assigned_shards)
            random.shuffle(assigned_shards)

        for shard_path in assigned_shards:
            batches_in_shard = torch.load(shard_path)
            for batch in batches_in_shard:
                images, labels = batch["images"], batch["labels"]
                for image, label in zip(images, labels):
                    yield image, label

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        
        # Partition shards across workers
        if worker_info is None:
            assigned_shards = list(self.shard_paths)
        else:
            worker_id = worker_info.id
            num_workers = worker_info.num_workers
            assigned_shards = self.shard_paths[worker_id::num_workers]

        stream = self._sample_generator(assigned_shards)

        if self.buffer_size <= 1:
            for sample in stream:
                yield sample
            return

        # Buffer shuffling
        buffer = []
        for sample in stream:
            buffer.append(sample)
            if len(buffer) >= self.buffer_size:
                rand_idx = random.randint(0, len(buffer) - 1)
                yield buffer.pop(rand_idx)

        random.shuffle(buffer)
        for sample in buffer:
            yield sample

# Create Dataset and DataLoader
buffered_dataset = BufferedShardedDataset(
    shard_paths=shard_files,
    shuffle_shards=True,
    buffer_size=1000
)

buffered_loader = DataLoader(
    buffered_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
)

first_images, first_labels = next(iter(buffered_loader))
print("First batch labels after buffer shuffling:")
print(first_labels)

## Step 6: Feature Extraction & Linear SVM Training
Extracting image features directly from the streaming sharded DataLoader and training an SVM model with `scikit-learn`.

In [ ]:
print("Extracting features from sharded loader...")
x_data = []
y_data = []

for images, labels in buffered_loader:
    flattened_images = images.view(images.size(0), -1).numpy()
    x_data.append(flattened_images)
    y_data.append(labels.numpy())

X = np.vstack(x_data)
y = np.concatenate(y_data)

# Split into Train and Validation sets (80/20 split)
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Dataset extracted: {X_train.shape[0]} training samples, {X_val.shape[0]} validation samples.")

# Initialize and train Linear Support Vector Classifier (LinearSVC)
print("Training Linear SVM model...")
svm_model = LinearSVC(max_iter=2000, random_state=42)

start_time = time.perf_counter()
svm_model.fit(X_train, y_train)
train_time = time.perf_counter() - start_time

# Evaluate
y_pred = svm_model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("================ SVM Training Results ================")
print(f"Training Time : {train_time:.2f} seconds")
print(f"Accuracy      : {accuracy * 100:.2f}%")
print("======================================================")
print("\nClassification Report:\n", classification_report(y_val, y_pred))